# Team 34 — V3.4 Stable, bản hợp nhất tối ưu và kiểm tra lỗi

**File độc lập: import notebook này lên Kaggle, bật GPU và Internet, chạy từ Cell 1.**
Tên phiên bản V3.4 theo yêu cầu; giữ các sửa chất lượng của nhánh V5 Balanced, không ghi đè notebook cũ.

## Cấu hình
YOLO26n-seg 11 lớp, train chính 65 epochs/batch16/mask_ratio4; teacher 15 epochs; fine-tune tối đa 8 epochs/batch8/mask_ratio2. Giữ 640/FP32/opset17 theo hợp đồng BTC. Ước lượng thời gian chưa được đo trên Kaggle; không bảo đảm điểm cao hơn khi chưa chạy đối chiếu.

## Xử lý các lỗi đã xác định
- Kiểm tra đúng phiên bản các gói và CUDA ngay từ đầu, trước khi xử lý dữ liệu. Log pip và environment được giữ trong outputs.
- Kagglehub có thể trả thư mục `/kaggle/input` chỉ đọc: tuyệt đối không giải nén hoặc xóa bên trong đó. ZIP được giải nén sang scratch riêng, kiểm tra đường dẫn và dung lượng trước.
- Chỉ tải SAM2 khi gặp nguồn bbox; truyền BGR đúng API; giải phóng VRAM trước teacher/train.
- Retry có giới hạn cho lỗi mạng tạm thời; không lặp lại 401/403/404 hay che lỗi dữ liệu. Nguồn tải lỗi được ghi source_failures.json. Thiếu nguồn đầy đủ/GT tối thiểu phải dừng để tránh train sai.
- Cache conversion chỉ dùng khi fingerprint và cấu hình khớp. Cache chưa hoàn tất không được xem là thành công. Rerun sau khi label bị thay đổi sẽ chuyển đổi lại từ nguồn, không tái sử dụng label giả như GT.
- CUDA OOM: giảm batch có giới hạn, giải phóng model/tensor trước lần thử tiếp; lỗi khác được báo riêng. Checkpoint mỗi 5 epochs; không vô hiệu hóa validator để cố xuất file.
- Nếu teacher lỗi, loại nguồn partial-label khỏi train; không giữ ảnh Sign-only làm nền âm. Nếu fine-tune/biểu đồ lỗi, giữ submission chính đã kiểm định. Official scoring của main vẫn là bước bắt buộc.
- Xuất `recovery_main_submission.zip` đã qua validator trước công việc phụ. `checkpoint_selection.json` chứng minh checkpoint nào thắng trên cùng holdout.

## Rerun và phục hồi
Đường dẫn tuyệt đối nằm dưới `/kaggle/working/team34_v34`. Đồng hồ không đặt lại khi rerun Cell 1 trong cùng kernel; mở kernel mới để bắt đầu một phiên mới. Nếu một nguồn mạng lỗi, xem source_failures.json, attach nguồn được giải nén sẵn và rerun các cell dữ liệu theo thứ tự. Không chạy train bỏ qua cổng dữ liệu.
Giữ master/prepared_v34.zip và thư mục runs để phục hồi một phiên bị ngắt. Cache chỉ tồn tại nếu ổ working còn được giữ; notebook không tự khôi phục dữ liệu của phiên Kaggle đã bị hủy.

## Giới hạn của kiểm chứng
Test local và chạy mô phỏng có thể chứng minh đường dẫn/chọn model/fallback, nhưng không chứng minh truy cập Kaggle API, CUDA thực, đủ bộ nhớ trên mọi GPU hoặc chất lượng AP. Không có cam kết “không bao giờ lỗi”. Nếu thiếu quyền dữ liệu, không đủ đĩa hoặc lớp GT, notebook dừng sớm với log thay vì train hoặc nộp một model không hợp lệ.
Ngân sách 7,5 giờ là cơ chế dừng train, không phải giới hạn cứng cho tải/giải nén/validation. Giữ 45 phút export; mạng hoặc dữ liệu chậm vẫn có thể vượt thời lượng phiên.

## Nguồn kỹ thuật
- [Kagglehub: Kaggle integration/read-only shared resources](https://github.com/Kaggle/kagglehub/blob/main/README.md)
- [Ultralytics 8.4.214 SAM preprocessing](https://raw.githubusercontent.com/ultralytics/ultralytics/v8.4.214/ultralytics/models/sam/predict.py)
- [Ultralytics training](https://docs.ultralytics.com/modes/train/)
- [FixMatch: confidence/consistency motivation](https://arxiv.org/abs/2001.07685) — không phải triển khai đầy đủ FixMatch.


In [ ]:
#@title 1. V3.4 Stable: cấu hình duy nhất và đường dẫn tuyệt đối
import os
import sys
import time
from pathlib import Path
TEAM_NAME, TEAM_ID = "Đội 34 – Nguyễn Bá Duy", 39
IS_KAGGLE = Path("/kaggle/working").exists()
RUN_STARTED = globals().get("V34_RUN_STARTED", time.monotonic())
V34_RUN_STARTED = RUN_STARTED
RUN_ROOT = (Path("/kaggle/working/team34_v34") if IS_KAGGLE else Path("work_kaggle_v34")).resolve()
REPO_DIR, TMP_RAW_DIR = RUN_ROOT / "repo", RUN_ROOT / "scratch"
STAGED_DIR, GOLDEN_VAL_DIR = RUN_ROOT / "staged", RUN_ROOT / "golden_val"
MASTER_DIR, OUTPUTS_DIR = RUN_ROOT / "master", RUN_ROOT / "outputs"
OUT_ZIP_PATH = OUTPUTS_DIR / "prepared_v34.zip"
for directory in (REPO_DIR, TMP_RAW_DIR, STAGED_DIR, OUTPUTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["KAGGLEHUB_CACHE"] = str(TMP_RAW_DIR / "kh_cache")
INSTALL_DEPS, REQUIRE_GPU = True, True
RUN_KAGGLE_INPUT_PUBLIC = True
AUTO_DOWNLOAD_KAGGLE_SLUGS = [
    "kaggleprollc/mapillary-vistas-image-data-collection",
    "a7madmostafa/bdd100k-yolo",
    "lvnduy/street-traffic-signs-in-vietnam-yolov8",
]
SKIP_STOCK_INPUT_SLUGS = ["vehicle-dataset-for-yolo", "obstacle-detection-dataset", "hackathon", "master_v", "team34_v34"]
MAPILLARY_MAX_IMAGES, BDD_MAX_IMAGES = 2800, 600
KAGGLE_MAX_IMAGES_PER_SOURCE, VAL_MAX_IMAGES_PER_SOURCE = 500, 240
SAM2_MODEL = "sam2_b.pt"
RUN_PARTIAL_LABEL_COMPLETION, RUN_STAGE1_MAPILLARY_TEACHER = True, True
STAGE1_EPOCHS, TEACHER_WEIGHTS = 15, "yolo11x-seg.pt"
PSEUDO_MIN_CONF, QUARANTINE_TEACHER_DISAGREEMENT = .50, True
RUN_CROP_ZOOM, RUN_COPY_PASTE, RUN_WEATHER_AUG = True, True, True
MAX_EXTRA_CROPS, COPY_PASTE_COUNT, WEATHER_COUNT = 400, 200, 75
AUTO_GOLDEN_VAL_TARGET, GOLDEN_MIN_PER_CLASS = 320, 25
MIN_RARE_TRAIN_INSTANCES, SANITY_IMAGES_PER_SOURCE = 200, 48
RUN_SANITY_GATE, FAIL_ON_SANITY_ALERT = True, True
RUN_TRAIN_IN_THIS_NOTEBOOK, RUN_CLEAN_REFINEMENT = True, True
TRAIN_EPOCHS, TRAIN_BATCH, TRAIN_PATIENCE, REFINE_EPOCHS = 65, 16, 15, 8
TOTAL_BUDGET_HOURS, EXPORT_RESERVE_HOURS = 7.5, .75
TEACHER_CAP_HOURS, MAIN_CAP_HOURS, REFINE_CAP_HOURS = .65, 3., .35
RUN_BENCHMARK_AND_QC, RUN_LOCAL_TRAIN_QC = True, False
QC_TRAIN_SAMPLE_IMAGES, LOCAL_PACK_MAX_VAL_IMAGES = 500, 60
RUN_NAME, BASELINE_SUMMARY_JSON = "team34_v34_stable", ""
print("V3.4 Stable | outputs:", OUTPUTS_DIR, flush=True)


In [ ]:
#@title 2. Cài đặt gói chuẩn & giải nén mã nguồn nhúng vào REPO_DIR
import base64
import importlib.metadata
import json
import shutil
import subprocess
import sys
import zlib

_BUNDLE_B64 = (
'...'
)
EMBEDDED_FILES = json.loads(zlib.decompress(base64.b64decode(_BUNDLE_B64)).decode("utf-8"))

for rel_path, content in EMBEDDED_FILES.items():
    target = REPO_DIR / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

print(f"Đã giải nén {len(EMBEDDED_FILES)} tệp mã nguồn V3.4 Stable vào: {REPO_DIR.resolve()}", flush=True)


# Install before importing binary/ML packages; fail here rather than hours later at export.
import importlib.metadata
import subprocess
import tempfile
import platform
if sys.version_info < (3,11):
    raise RuntimeError("The competition-pinned onnxruntime requires Python >=3.11; choose a current Kaggle image")
PINNED = {"ultralytics":"8.4.214", "onnx":"1.23.2", "onnxslim":"0.1.99", "onnxruntime":"1.32.0"}
if INSTALL_DEPS:
    mismatches = []
    for package, version in PINNED.items():
        try:
            installed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            installed = None
        if installed != version:
            if package in sys.modules:
                raise RuntimeError(f"{package} already imported at {installed}; restart the kernel before installing {version}")
            mismatches.append(f"{package}=={version}")
    specs = mismatches + ["pycocotools", "shapely", "huggingface_hub", "datasets", "kagglehub", "pyyaml", "matplotlib", "ipython"]
    if specs:
        pip = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--retries", "3", "--timeout", "60", *specs], capture_output=True, text=True, timeout=900)
        (OUTPUTS_DIR / "pip_install.log").write_text(pip.stdout + pip.stderr, encoding="utf-8")
        if pip.returncode:
            raise RuntimeError("Dependency installation failed; inspect outputs/pip_install.log and enable Internet")
    importlib.invalidate_caches()
versions = {package: importlib.metadata.version(package) for package in PINNED}
if versions != PINNED:
    raise RuntimeError(f"Pinned package mismatch: {versions}; restart kernel and run setup first")
for package in ("cv2", "numpy", "shapely", "pycocotools.mask", "onnx", "onnxruntime", "ultralytics"):
    importlib.import_module(package)
import torch
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before downloading/preparing data")
DEVICE = "0" if torch.cuda.is_available() else "cpu"
if DEVICE == "0":
    torch.zeros(1, device="cuda")  # Verify CUDA actually initializes, not merely enumerates a device.
    torch.cuda.empty_cache()
TEMP_FILES_DIR = TMP_RAW_DIR / "tempfiles"
TEMP_FILES_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(TEMP_FILES_DIR)
tempfile.tempdir = str(TEMP_FILES_DIR)
repo_str = str(REPO_DIR.resolve())
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
os.chdir(REPO_DIR)
from src.stable_v34 import atomic_json, release_gpu
atomic_json(OUTPUTS_DIR / "environment.json", {"python":platform.python_version(), "versions":versions,
    "torch":torch.__version__, "device":DEVICE,
    "gpu":torch.cuda.get_device_name(0) if DEVICE == "0" else None})
print("V3.4 preflight passed:", versions, DEVICE, flush=True)


In [ ]:
#@title 3. Dò /kaggle/input (chống bọc thư mục datasets/) & Chuẩn bị hàng đợi Kagglehub
from src.kaggle_adapter import detect_dataset_format, discover_all_kaggle_inputs

DISCOVERED_KAGGLE = []
KAGGLEHUB_QUEUE = []

if RUN_KAGGLE_INPUT_PUBLIC:
    skip_slugs = {s.lower() for s in SKIP_STOCK_INPUT_SLUGS}
    mounted = discover_all_kaggle_inputs(Path("/kaggle/input"), skip_names=skip_slugs)
    for info in mounted:
        nm_low = f"{info.get('name', '')} {info.get('images_dir', '')} {info.get('labels_path', '')}".lower()
        if any(sk in nm_low for sk in skip_slugs):
            print(f"[SKIP] Bỏ qua nguồn cũ/stock không dùng cho V3: {info.get('name')}")
            continue
        DISCOVERED_KAGGLE.append(info)

    mounted_text = " ".join(f"{d.get('name', '')} {d.get('images_dir', '')}".lower() for d in DISCOVERED_KAGGLE)
    for slug in AUTO_DOWNLOAD_KAGGLE_SLUGS:
        slug_short = slug.split("/")[-1].lower()
        if slug_short in mounted_text:
            print(f"[SKIP] Đã gắn sẵn trong /kaggle/input: {slug}")
        else:
            KAGGLEHUB_QUEUE.append(slug)

print(f"Đã gắn sẵn: {len(DISCOVERED_KAGGLE)} | Sẽ tải-chuyển đổi-xóa cache tuần tự ở Cell 5: {len(KAGGLEHUB_QUEUE)}")
for s in KAGGLEHUB_QUEUE:
    print("  - https://www.kaggle.com/datasets/" + s)


In [ ]:
#@title 4. Khởi tạo staging; chỉ tải SAM2 khi thật sự cần bbox -> mask
from src.stable_v34 import LazySAM
STAGED_SOURCES, CLEAN_VAL_SOURCES, CLEAN_TRAIN_SOURCES = [], [], []
SAM_PREDICTOR = LazySAM(SAM2_MODEL, DEVICE)
SOURCE_FAILURES = []


In [ ]:
#@title 5. Tải có retry, hỗ trợ input chỉ đọc và cache conversion có kiểm chứng
import kagglehub
from src.kaggle_adapter import detect_dataset_format, detect_val_split, auto_convert_detected_dataset
from src.stable_v34 import retry_download, unpack_readonly_input, cached_stage, atomic_json

for index, slug in enumerate(AUTO_DOWNLOAD_KAGGLE_SLUGS):
    short = slug.split("/")[-1]
    try:
        mounted = next((info for info in DISCOVERED_KAGGLE if short.replace("-", "_") in str(info).lower().replace("-", "_")), None)
        info = mounted
        if info is None:
            source_root = Path(retry_download(lambda: kagglehub.dataset_download(slug)))
            info = detect_dataset_format(source_root)
            if info is None:
                extracted = unpack_readonly_input(source_root, TMP_RAW_DIR / "extracted" / short)
                info = detect_dataset_format(extracted)
        if info is None:
            raise ValueError("Dataset format not detected; attach an extracted, annotated dataset")
        info = dict(info, name=slug.replace("/", "__"), url="https://www.kaggle.com/datasets/"+slug)
        safe = info["name"].replace("-", "_")
        full = any(marker in safe for marker in ("mapillary", "bdd100k", "cityscapes"))
        cap = MAPILLARY_MAX_IMAGES if "mapillary" in safe else BDD_MAX_IMAGES if "bdd100k" in safe else KAGGLE_MAX_IMAGES_PER_SOURCE
        out = STAGED_DIR / f"kaggle_{index:02d}_{safe}"
        signature = {"recipe":"3.4", "source":slug, "descriptor":info, "cap":cap, "seed":TEAM_ID}
        result = cached_stage(out, signature, lambda destination: auto_convert_detected_dataset(
            info, destination, sam_predictor=SAM_PREDICTOR, max_images=cap, seed=TEAM_ID, split="train",
            registry_path=REPO_DIR / "datasets_registry.md"))
        STAGED_SOURCES.append(out)
        if full:
            CLEAN_TRAIN_SOURCES.append(out)
            val_info = detect_val_split(info)
            if val_info:
                val_out = STAGED_DIR / f"_val_{index:02d}_{safe}"
                val_cap = VAL_MAX_IMAGES_PER_SOURCE if "mapillary" in safe else 90
                val_signature = dict(signature, descriptor=val_info, cap=val_cap, split="val")
                cached_stage(val_out, val_signature, lambda destination: auto_convert_detected_dataset(
                    val_info, destination, sam_predictor=SAM_PREDICTOR, max_images=val_cap,
                    seed=TEAM_ID, split="val", registry_path=None))
                CLEAN_VAL_SOURCES.append(val_out)
        print("[SOURCE OK]", slug, result, flush=True)
    except Exception as exc:
        failure = {"source":slug, "error_type":type(exc).__name__, "message":str(exc)}
        SOURCE_FAILURES.append(failure)
        print("[SOURCE FAILED]", failure, flush=True)
    finally:
        atomic_json(OUTPUTS_DIR / "source_failures.json", SOURCE_FAILURES)
        # Never delete mounted /kaggle/input resources or unrelated user caches.
        owned_cache = TMP_RAW_DIR / "kh_cache"
        if owned_cache.is_dir():
            shutil.rmtree(owned_cache)
SAM_PREDICTOR.release()
if not CLEAN_TRAIN_SOURCES:
    raise RuntimeError("No fully annotated street source available. See source_failures.json; attach Mapillary/BDD and rerun data cells")


In [ ]:
#@title 6. Giải phóng SAM2 trước teacher/train
SAM_PREDICTOR.release()
print("SAM2 VRAM released", flush=True)


In [ ]:
#@title 7. Khóa Golden Val: quota theo mẫu được chấm + manifest nguồn + chống rò trước augment
from src.data_gates import build_golden_holdout, exclude_golden_from_staged_train

shutil.rmtree(GOLDEN_VAL_DIR, ignore_errors=True)
holdout = build_golden_holdout(
    CLEAN_VAL_SOURCES, CLEAN_TRAIN_SOURCES, GOLDEN_VAL_DIR,
    min_per_class=GOLDEN_MIN_PER_CLASS, target_images=AUTO_GOLDEN_VAL_TARGET, seed=TEAM_ID,
)
print(json.dumps({k: v for k, v in holdout.items() if k != "manifest"}, indent=2, ensure_ascii=False))
shutil.copy2(GOLDEN_VAL_DIR / "holdout_manifest.json", OUTPUTS_DIR / "holdout_manifest.json")
if not holdout["ok"]:
    raise RuntimeError(f"Golden Val thiếu mẫu được chấm; bổ sung nguồn nhãn người trước train: {holdout['missing_instances']}")
# Chỉ xóa bản staged dùng tạm, không sửa dữ liệu upstream. Loại cả bản trùng ở nguồn khác trước khi hoàn thiện nhãn & augment.
print("Removed staged train/val overlap:", exclude_golden_from_staged_train(STAGED_SOURCES, GOLDEN_VAL_DIR))


In [ ]:
#@title 7B. Bổ sung nhãn VN; loại nguồn partial nếu teacher không hoàn tất
try:
    #@title 7B. Teacher chất lượng: hai góc nhìn + giữ GT + cách ly bất đồng
    import gc
    import torch
    from ultralytics import YOLO
    from src.build_master import build_master_dataset
    from src.stable_v34 import train_resilient, release_gpu, atomic_json
    from src.balanced_budget import phase_hours, StopAtDeadline
    from src.prepare_vn_frames import UltralyticsTeacherPredictor
    from src.score_focus_v5 import complete_labels_with_consensus, quarantine_uncertain

    vn_partial_sources = [s for s in STAGED_SOURCES if any(k in s.name.lower() for k in ("vnts", "lvnduy", "traffic_sign"))]
    coco_sources = [s for s in STAGED_SOURCES if "coco" in s.name.lower()]
    full_sources_for_stage1 = [s for s in STAGED_SOURCES if any(k in s.name.lower() for k in ("mapillary", "bdd100k", "cityscapes"))]
    completion_reports = {}
    if RUN_PARTIAL_LABEL_COMPLETION:
        if vn_partial_sources:
            teacher_coco = UltralyticsTeacherPredictor(weights=TEACHER_WEIGHTS, conf=0.25, imgsz=1280, device=DEVICE)
            for source in vn_partial_sources:
                result = complete_labels_with_consensus(
                    source, teacher_coco, preserved_classes={9}, fill_classes={0,1,2,4,5,6,7,8,10},
                    min_conf=PSEUDO_MIN_CONF, per_class_min_conf={1: .55, 2: .55, 4: .45, 6: .45, 8: .45, 10: .45})
                completion_reports.setdefault(str(source), []).append(result)
                print(source.name, {k:v for k,v in result.items() if k != "records"}, flush=True)
            del teacher_coco
            gc.collect()
            torch.cuda.empty_cache()

        if RUN_STAGE1_MAPILLARY_TEACHER and full_sources_for_stage1 and (vn_partial_sources or coco_sources):
            stage1_data_dir = TMP_RAW_DIR / "stage1_master"
            shutil.rmtree(stage1_data_dir, ignore_errors=True)
            stage1_summary = build_master_dataset(full_sources_for_stage1, GOLDEN_VAL_DIR, stage1_data_dir)
            if not stage1_summary["qc_ok"]:
                raise RuntimeError("Stage-1 dataset failed QC")
            s1_proj = RUN_ROOT / "runs_teacher"
            teacher_hours = phase_hours(RUN_STARTED, TOTAL_BUDGET_HOURS, EXPORT_RESERVE_HOURS + 1.5, TEACHER_CAP_HOURS)
            if teacher_hours < .10:
                raise RuntimeError("Insufficient time for optional teacher training")
            teacher_end = time.monotonic() + teacher_hours*3600
            teacher_kwargs = dict(data=str(stage1_data_dir / "data.yaml"), imgsz=640, epochs=STAGE1_EPOCHS,
                batch=min(8,TRAIN_BATCH), seed=TEAM_ID, device=DEVICE, workers=2, cache=False,
                optimizer="AdamW", lr0=.001, cos_lr=True, mask_ratio=4, save_period=5,
                mosaic=.5, mixup=0., copy_paste=0., close_mosaic=5, max_det=100,
                project=str(s1_proj), name="stage1_teacher", exist_ok=True)
            s1_best, teacher_attempts = train_resilient(YOLO, "yolo26s-seg.pt", teacher_kwargs,
                lambda: StopAtDeadline(max(.001,(teacher_end-time.monotonic())/3600)))
            atomic_json(OUTPUTS_DIR / "teacher_train_attempts.json", teacher_attempts)
            shutil.copy2(s1_best, OUTPUTS_DIR / "stage1_teacher.pt")
            gc.collect()
            torch.cuda.empty_cache()
            teacher_map = UltralyticsTeacherPredictor(weights=str(OUTPUTS_DIR / "stage1_teacher.pt"), conf=.25, imgsz=1280, device=DEVICE)
            for source, preserve, fill in [(s, {0,1,2,4,5,6,7,8,10}, {3,9}) for s in coco_sources] + [(s, {0,1,2,4,5,6,7,8,9,10}, {3}) for s in vn_partial_sources]:
                result = complete_labels_with_consensus(source, teacher_map, preserve, fill,
                    source_mapping_name="competition_direct", min_conf=PSEUDO_MIN_CONF,
                    per_class_min_conf={3:.60, 9:.50})
                completion_reports.setdefault(str(source), []).append(result)
                print(source.name, {k:v for k,v in result.items() if k != "records"}, flush=True)
            del teacher_map
            gc.collect()
            torch.cuda.empty_cache()

        quarantine_reports = []
        if QUARANTINE_TEACHER_DISAGREEMENT:
            for source_text, reports in completion_reports.items():
                quarantine_reports.append(quarantine_uncertain(Path(source_text), reports, TMP_RAW_DIR / "teacher_quarantine"))
        (OUTPUTS_DIR / "teacher_consensus_summary.json").write_text(json.dumps({
            "sources": {k: [{a:b for a,b in r.items() if a != "records"} for r in reports] for k,reports in completion_reports.items()},
            "quarantine": quarantine_reports}, indent=2), encoding="utf-8")
        STAGED_SOURCES = [s for s in STAGED_SOURCES if any((s / "images/train").glob("*"))]
    elif vn_partial_sources:
        raise RuntimeError("Partial Sign-only datasets require annotation completion before training")

except Exception as exc:
    from src.stable_v34 import atomic_json, release_gpu
    atomic_json(OUTPUTS_DIR / "teacher_completion_failure.json", {"error_type":type(exc).__name__, "message":str(exc), "action":"exclude every partial-label source from training"})
    STAGED_SOURCES = [s for s in STAGED_SOURCES if not any(k in s.name.lower() for k in ("vnts", "lvnduy", "traffic_sign", "coco"))]
    print("[OPTIONAL PARTIAL SOURCES EXCLUDED]", exc, flush=True)
finally:
    for _name in ("teacher_coco", "teacher_map", "s1_model"):
        globals().pop(_name, None)
    from src.stable_v34 import release_gpu
    release_gpu()


In [ ]:
#@title 8. Tăng mẫu lớp yếu từ nhãn đầy đủ, giới hạn ảnh tổng hợp
from src.augment_and_crop import generate_crop_zoom_dataset, generate_copypaste_dataset, generate_weather_dataset
from src.score_focus_v5 import cap_generated_images

base_sources = [s for s in STAGED_SOURCES if not s.name.startswith("aug_")]
human_full_sources = [s for s in base_sources if any(k in s.name for k in ("mapillary", "bdd100k", "cityscapes"))]
if not human_full_sources:
    raise RuntimeError("V5 requires fully annotated original street sources")

# The bank/background use full-label sources, avoiding amplification of teacher mistakes.
if RUN_CROP_ZOOM:
    for idx, source in enumerate(human_full_sources):
        out = STAGED_DIR / f"aug_weakclass_crops_{idx:02d}"
        shutil.rmtree(out, ignore_errors=True)
        result = generate_crop_zoom_dataset(source, out, target_classes=(3,5,6,7,9,10),
            max_crops_per_image=1, min_target_area_640=32., max_target_area_640=1800., seed=TEAM_ID+idx)
        kept = cap_generated_images(out, MAX_EXTRA_CROPS // len(human_full_sources), TEAM_ID+idx)
        if kept:
            STAGED_SOURCES.append(out)
        print(source.name, "weak-class crops", kept, flush=True)
if RUN_COPY_PASTE:
    out = STAGED_DIR / "aug_copypaste"
    shutil.rmtree(out, ignore_errors=True)
    result = generate_copypaste_dataset(human_full_sources, human_full_sources, out,
        rare_classes=(3,5,6,7), n_output=COPY_PASTE_COUNT, pastes_per_image=(1,2), seed=TEAM_ID)
    if result["images_written"]:
        STAGED_SOURCES.append(out)
    print(result, flush=True)
if RUN_WEATHER_AUG:
    out = STAGED_DIR / "aug_weather"
    shutil.rmtree(out, ignore_errors=True)
    result = generate_weather_dataset(human_full_sources[0], out, max_images=WEATHER_COUNT, seed=TEAM_ID)
    if result["images_written"]:
        STAGED_SOURCES.append(out)


In [ ]:
#@title 9. Gộp Master V3.4, Kiểm tra Chống Đầu Độc Nhãn Âm & Sanity Gate Trước Khi Train
import torch
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("V3.4 requires a GPU; do not start score-focused training on CPU")
from IPython.display import Image as IPyImage, display
from src.build_master import build_master_dataset
from src.eval_golden import render_per_class_preview_grid, run_pretrain_semantic_sanity_check
from src.validate_dataset import load_competition_config, print_qc_summary, run_qc

# Cổng Chống Đầu Độc Nhãn Âm: mọi nguồn gốc không phải aug_ phải có >= 4 lớp nhãn (cấm nguồn 1-lớp lọt vào train!)
for _src in STAGED_SOURCES:
    if _src.name.startswith("aug_"):
        continue
    _present_cls = set()
    for _lf in (_src / "labels" / "train").glob("*.txt"):
        for _ln in _lf.read_text(encoding="utf-8", errors="ignore").splitlines():
            _tok = _ln.split()
            if _tok:
                _present_cls.add(int(_tok[0]))
        if len(_present_cls) >= 4:
            break
    print(f"  [COVERAGE CHECK] {_src.name}: >= {len(_present_cls)} lớp", flush=True)
    if len(_present_cls) < 4:
        print(f"[REVIEW] Nguồn {_src.name}: {len(_present_cls)} lớp; kiểm tra completeness theo ảnh")

shutil.rmtree(MASTER_DIR, ignore_errors=True)
master_summary = build_master_dataset(
    train_sources=STAGED_SOURCES,
    golden_val_dir=GOLDEN_VAL_DIR,
    master_dir=MASTER_DIR,
    zip_out=OUT_ZIP_PATH,
)
print("Kết quả Build Master V3.4:", json.dumps(master_summary, indent=2, ensure_ascii=False))

cfg = load_competition_config(REPO_DIR / "configs/competition.yaml")
qc_rep = run_qc(MASTER_DIR, cfg, out_dir=OUTPUTS_DIR / "qc_master")
print_qc_summary(qc_rep, cfg)

preview_grid_p = OUTPUTS_DIR / "preview_per_class.jpg"
render_per_class_preview_grid(MASTER_DIR, preview_grid_p, split="train", samples_per_class=4, seed=TEAM_ID)
if preview_grid_p.is_file():
    if IS_KAGGLE:
        shutil.copy2(preview_grid_p, Path("/kaggle/working/preview_per_class.jpg"))
    display(IPyImage(filename=str(preview_grid_p)))

# Coverage and Tier-1 QC are mandatory, independent of semantic gate flags.
if not qc_rep["ok"]:
    raise RuntimeError("Tier-1 QC failed; inspect qc_master reports before training")
rare_shortfalls = {
    cfg["classes"][c]: qc_rep["per_class"][cfg["classes"][c]]["by_split"].get("train", 0)
    for c in (6, 7)
    if qc_rep["per_class"][cfg["classes"][c]]["by_split"].get("train", 0) < MIN_RARE_TRAIN_INSTANCES
}
if rare_shortfalls:
    raise RuntimeError(f"Train thiếu Bicycle/Cyclist (minimum={MIN_RARE_TRAIN_INSTANCES}): {rare_shortfalls}")

if RUN_SANITY_GATE:
    source_sanity = {}
    for source in STAGED_SOURCES:
        if source.name.startswith("aug_"):
            continue
        sanity = run_pretrain_semantic_sanity_check(
            source, split="train", sample_images=SANITY_IMAGES_PER_SOURCE, seed=TEAM_ID, weights="yolo26n-seg.pt")
        source_sanity[source.name] = sanity
        print(source.name, json.dumps(sanity, indent=2, ensure_ascii=False))
    (OUTPUTS_DIR / "source_semantic_sanity.json").write_text(json.dumps(source_sanity, indent=2), encoding="utf-8")
    blocked = {name: result for name, result in source_sanity.items() if result["ok"] is False}
    if blocked and FAIL_ON_SANITY_ALERT:
        raise RuntimeError(f"Sanity Gate failed or lacked evidence for sources: {list(blocked)}")

shutil.copy2(REPO_DIR / "datasets_registry.md", OUTPUTS_DIR / "datasets_registry.md")
print(f"\n[XONG DATA V3.4] {OUT_ZIP_PATH} ({OUT_ZIP_PATH.stat().st_size / 2**20:.1f} MiB)")

if qc_rep["duplicates"]["leak_exact"] or qc_rep["duplicates"]["leak_near"]:
    raise RuntimeError("Train/Val leakage found; do not train")

from src.stable_v34 import atomic_json, dataset_fingerprint
atomic_json(OUTPUTS_DIR / "dataset_manifest.json", {"recipe":"3.4-stable", "seed":TEAM_ID,
    "fingerprint":dataset_fingerprint(MASTER_DIR), "master_summary":master_summary,
    "source_failures":SOURCE_FAILURES})


In [ ]:
#@title 10. Train phục hồi OOM, xuất dự phòng, chấm chính thức rồi chọn checkpoint
from ultralytics import YOLO
from src.stable_v34 import train_resilient, release_gpu, atomic_json
from src.balanced_budget import phase_hours, StopAtDeadline
from src.score_focus_v5 import make_repeat_factor_dataset, holdout_fingerprint, choose_official_candidate
from src.export_verify_onnx import export_and_verify
from src.eval_golden import run_eval_or_qc

BEST_PT_PATH = SUBMISSION_ZIP_PATH = SELECTED_TRAIN_DIR = None
if RUN_TRAIN_IN_THIS_NOTEBOOK:
    train_yaml = MASTER_DIR / "data_repeat.yaml"
    sampling = make_repeat_factor_dataset(MASTER_DIR, train_yaml, max_extra_fraction=.20, seed=TEAM_ID)
    print("[SAMPLING]", sampling, flush=True)
    run_proj = RUN_ROOT / "runs"
    common = dict(imgsz=640, batch=TRAIN_BATCH, device=DEVICE, workers=2, cache=False, seed=TEAM_ID,
        deterministic=True, amp=True, optimizer="AdamW", cos_lr=True, mask_ratio=4, overlap_mask=True,
        max_det=100, save_period=5, project=str(run_proj), exist_ok=True)
    main_hours = phase_hours(RUN_STARTED, TOTAL_BUDGET_HOURS, EXPORT_RESERVE_HOURS, MAIN_CAP_HOURS)
    if main_hours < .25:
        raise RuntimeError("Not enough time for meaningful training; keep master data and start a new GPU session")
    phase_end = time.monotonic() + main_hours*3600
    make_deadline = lambda: StopAtDeadline(max(.001, (phase_end-time.monotonic())/3600))
    kwargs = dict(common, data=str(train_yaml), epochs=TRAIN_EPOCHS, patience=TRAIN_PATIENCE,
        lr0=.0015, lrf=.02, weight_decay=.0005, warmup_epochs=3., scale=.5, degrees=2., hsv_v=.35,
        mosaic=.6, close_mosaic=12, copy_paste=.10, mixup=0., multi_scale=False, name=RUN_NAME+"_main")
    main_best, attempts = train_resilient(YOLO, "yolo26n-seg.pt", kwargs, make_deadline)
    atomic_json(OUTPUTS_DIR / "train_attempts.json", {"main":attempts})
    # Publish a validated recovery artifact before any optional work.
    main_dir = OUTPUTS_DIR / "candidates/main"
    exported = {"main": export_and_verify(main_best, main_dir, notebook_path=REPO_DIR / "train_and_export.ipynb",
        smoke_images_dir=MASTER_DIR / "images/val")}
    shutil.copy2(main_dir / "submission.zip", OUTPUTS_DIR / "recovery_main_submission.zip")
    shutil.copy2(main_dir / "submission.zip", OUTPUTS_DIR / "submission.zip")
    candidates = [("main", main_best)]
    refine_hours = phase_hours(RUN_STARTED, TOTAL_BUDGET_HOURS, EXPORT_RESERVE_HOURS, REFINE_CAP_HOURS)
    if RUN_CLEAN_REFINEMENT and refine_hours >= .10:
        try:
            refine_yaml = MASTER_DIR / "data_clean_refine.yaml"
            make_repeat_factor_dataset(MASTER_DIR, refine_yaml, clean_only=True, max_extra_fraction=.20, seed=TEAM_ID)
            phase_end = time.monotonic() + refine_hours*3600
            refine_kwargs = dict(common, mask_ratio=2, batch=min(TRAIN_BATCH,8), data=str(refine_yaml),
                epochs=REFINE_EPOCHS, patience=REFINE_EPOCHS, lr0=.00015, lrf=.1, warmup_epochs=1.,
                scale=.15, degrees=0., hsv_v=.15, mosaic=0., close_mosaic=0, copy_paste=0., mixup=0.,
                multi_scale=False, name=RUN_NAME+"_refine")
            refine_best, refine_attempts = train_resilient(YOLO, main_best, refine_kwargs, make_deadline)
            atomic_json(OUTPUTS_DIR / "refine_attempts.json", refine_attempts)
            candidates.append(("clean_refine", refine_best))
        except Exception as exc:
            atomic_json(OUTPUTS_DIR / "optional_refine_failure.json", {"error_type":type(exc).__name__, "message":str(exc), "fallback":"validated main checkpoint"})
            print("[REFINE SKIPPED]", exc, flush=True)
        finally:
            release_gpu()
    fingerprint = holdout_fingerprint(MASTER_DIR)
    evaluated = []
    for name, weights in candidates:
        candidate_dir = OUTPUTS_DIR / "candidates" / name
        try:
            if name not in exported:
                exported[name] = export_and_verify(weights, candidate_dir, notebook_path=REPO_DIR / "train_and_export.ipynb",
                    smoke_images_dir=MASTER_DIR / "images/val")
            evaluation = run_eval_or_qc(Path(exported[name]["onnx"]), MASTER_DIR / "images/val", MASTER_DIR / "labels/val",
                candidate_dir / "official_validation", mode="eval", notebook_path=REPO_DIR / "train_and_export.ipynb")
            candidate = {"name":name, "weights":str(weights), "export_dir":str(candidate_dir),
                "holdout_sha256":fingerprint, "n_images":evaluation["n_images"], "score_summary":evaluation["score_summary"]}
            choose_official_candidate([candidate])  # A fallback metric must never choose the submitted model.
            evaluated.append(candidate)
        except Exception as exc:
            atomic_json(candidate_dir / "evaluation_failure.json", {"error_type":type(exc).__name__, "message":str(exc)})
            if name == "main":
                raise RuntimeError("Official evaluation of main failed; recovery_main_submission.zip is validated but not scored") from exc
            print("[OPTIONAL CANDIDATE SKIPPED]", name, exc, flush=True)
    winner = choose_official_candidate(evaluated)
    atomic_json(OUTPUTS_DIR / "checkpoint_selection.json", {"selected":winner["name"], "candidates":evaluated})
    BEST_PT_PATH = Path(winner["weights"])
    SELECTED_TRAIN_DIR = BEST_PT_PATH.parent.parent
    shutil.copy2(BEST_PT_PATH, OUTPUTS_DIR / "best.pt")
    for filename in ("model.onnx", "submission.zip"):
        shutil.copy2(Path(winner["export_dir"]) / filename, OUTPUTS_DIR / filename)
    SUBMISSION_ZIP_PATH = OUTPUTS_DIR / "submission.zip"
    print("[VALIDATED AND SCORED]", winner["name"], winner["score_summary"]["score"], SUBMISSION_ZIP_PATH, flush=True)


In [ ]:
#@title 11. Báo cáo phụ, không thay đổi submission đã kiểm định
try:
    #@title 11. Xuất Báo cáo So sánh, Confident QC & Gói quick_local_pack.zip
    QUICK_PACK_PATH = OUTPUTS_DIR / "quick_local_pack.zip"
    sub_p = Path(SUBMISSION_ZIP_PATH) if SUBMISSION_ZIP_PATH else (OUTPUTS_DIR / "submission.zip")
    pt_p = Path(BEST_PT_PATH) if BEST_PT_PATH else (OUTPUTS_DIR / "best.pt")
    train_run_p = SELECTED_TRAIN_DIR

    if RUN_BENCHMARK_AND_QC and sub_p.is_file() and (MASTER_DIR / "images").is_dir():
        from src.benchmark_report import run_benchmark
        from src.eval_golden import run_eval_or_qc
        from src.selfcheck_bridge import materialize_selfcheck

        bench_dir = OUTPUTS_DIR / "benchmark" / RUN_NAME
        rep = run_benchmark(
            submission_or_onnx=sub_p,
            dataset_dir=MASTER_DIR,
            out_dir=bench_dir,
            run_name=RUN_NAME,
            weights_pt=pt_p if pt_p.is_file() else None,
            baseline_json=Path(BASELINE_SUMMARY_JSON) if BASELINE_SUMMARY_JSON else None,
            train_run_dir=train_run_p,
            notebook_path=REPO_DIR / "train_and_export.ipynb",
        )
        shutil.copy2(bench_dir / "benchmark_report.md", OUTPUTS_DIR / "benchmark_report.md")
        shutil.copy2(bench_dir / "benchmark_summary.json", OUTPUTS_DIR / "benchmark_summary.json")
        for img_art in list(bench_dir.glob("*.png")) + list(bench_dir.glob("*.jpg")):
            shutil.copy2(img_art, OUTPUTS_DIR / img_art.name)

        sc = materialize_selfcheck(REPO_DIR / "train_and_export.ipynb", Path("/kaggle/working/_selfcheck_eval") if IS_KAGGLE else Path("work_kaggle_v5_balanced/_selfcheck_eval"))
        zip_s = sc.package.inspect_zip(str(sub_p), sc.config)
        onnx_p = sc.package.extract_model(str(sub_p), OUTPUTS_DIR / "_tmp_extract", sc.config, zip_s)
        if RUN_LOCAL_TRAIN_QC:
            qc_rep = run_eval_or_qc(
                model_path=onnx_p,
                images_dir=MASTER_DIR / "images" / "train",
                labels_dir=MASTER_DIR / "labels" / "train",
                out_dir=bench_dir / "qc_confident_train",
                mode="qc-train",
                notebook_path=REPO_DIR / "train_and_export.ipynb",
                max_images=QC_TRAIN_SAMPLE_IMAGES,
                seed=TEAM_ID,
            )
            shutil.copy2(bench_dir / "qc_confident_train" / "confident_qc_report.json", OUTPUTS_DIR / "confident_qc_report.json")
            print("Confident QC suspects:", {k: len(v) for k, v in qc_rep["confident_qc"].items()})

    if sub_p.is_file() and (MASTER_DIR / "images").is_dir():
        from src.benchmark_report import build_quick_local_pack

        extra_docs = [
            p for p in [
                OUTPUTS_DIR / "benchmark_report.md",
                OUTPUTS_DIR / "benchmark_summary.json",
                OUTPUTS_DIR / "datasets_registry.md",
                OUTPUTS_DIR / "confident_qc_report.json",
                OUTPUTS_DIR / "preview_per_class.jpg",
                OUTPUTS_DIR / "holdout_manifest.json",
                OUTPUTS_DIR / "checkpoint_selection.json",
                OUTPUTS_DIR / "teacher_consensus_summary.json",
                OUTPUTS_DIR / "source_semantic_sanity.json",
                Path(BASELINE_SUMMARY_JSON) if BASELINE_SUMMARY_JSON else None,
            ]
            if p and Path(p).is_file()
        ]
        build_quick_local_pack(
            out_zip=QUICK_PACK_PATH,
            submission_or_onnx=sub_p,
            dataset_dir=MASTER_DIR,
            weights_pt=pt_p if pt_p.is_file() else None,
            train_run_dir=train_run_p,
            extra_files=extra_docs,
            competition_path=REPO_DIR / "configs/competition.yaml",
            max_val_images=LOCAL_PACK_MAX_VAL_IMAGES,
            seed=TEAM_ID,
        )
        if IS_KAGGLE:
            shutil.copy2(sub_p, Path("/kaggle/working/submission.zip"))
            if (OUTPUTS_DIR / "benchmark_report.md").is_file():
                shutil.copy2(OUTPUTS_DIR / "benchmark_report.md", Path("/kaggle/working/benchmark_report.md"))
        print(f"[XONG PACK TEST LAPTOP] {QUICK_PACK_PATH} ({QUICK_PACK_PATH.stat().st_size / 2**20:.2f} MiB)")

except Exception as exc:
    from src.stable_v34 import atomic_json
    atomic_json(OUTPUTS_DIR / "optional_report_failure.json", {"error_type":type(exc).__name__, "message":str(exc)})
    print("[REPORT OPTIONAL]", exc, "Validated submission remains in", OUTPUTS_DIR, flush=True)


In [ ]:
#@title 12. Tải submission và chẩn đoán
from IPython.display import FileLink, display
os.chdir(Path("/kaggle/working") if IS_KAGGLE else RUN_ROOT)
for filename in ("submission.zip", "recovery_main_submission.zip", "checkpoint_selection.json", "best.pt",
                 "environment.json", "source_failures.json", "train_attempts.json", "dataset_manifest.json",
                 "holdout_manifest.json", "benchmark_report.md", "optional_refine_failure.json",
                 "teacher_completion_failure.json", "prepared_v34.zip"):
    path = OUTPUTS_DIR / filename
    if path.is_file():
        display(FileLink(os.path.relpath(path, Path.cwd())))
print("V3.4 output directory:", OUTPUTS_DIR)
